**Split Data**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Memuat dataset
df = pd.read_csv('Mental Health Burnout Tech.csv')

# 2. Pembersihan Data Dasar
# Menghapus employee_id karena tidak memiliki nilai prediktif
df = df.drop('employee_id', axis=1)

# 3. Menentukan Fitur (X) dan Target (y)
X = df.drop('burnout_level', axis=1)
y = df['burnout_level']

# 4. Membagi data menjadi Training dan Testing
# Stratify=y memastikan proporsi level burnout (Low/Moderate/High/Severe) seimbang di kedua set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Menampilkan distribusi target untuk verifikasi
print("Distribusi Burnout Level di data latih:")
print(y_train.value_counts(normalize=True))

Distribusi Burnout Level di data latih:
burnout_level
Severe      0.285762
Moderate    0.262550
Low         0.258075
High        0.193612
Name: proportion, dtype: float64


**Baseline**

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 1. Memilih hanya kolom numerik dan mengisi nilai kosong (jika ada)
X_train_base = X_train.select_dtypes(include=['int64', 'float64']).fillna(0)
X_test_base  = X_test.select_dtypes(include=['int64', 'float64']).fillna(0)

# 2. Inisialisasi dan Pelatihan Model
# Menggunakan multi_class='multinomial' (default LogisticRegression untuk > 2 kelas)
model_base = LogisticRegression(max_iter=2000, random_state=42)
model_base.fit(X_train_base, y_train)

# 3. Prediksi
pred_base = model_base.predict(X_test_base)
# Untuk multiclass, kita ambil seluruh probabilitas kelas untuk ROC-AUC
proba_base = model_base.predict_proba(X_test_base)

# 4. Evaluasi Metrik
print('=== BASELINE (Multiclass) ===')
print(f'Accuracy : {accuracy_score(y_test, pred_base):.4f}')

# Untuk multiclass, f1_score memerlukan parameter 'average' (contoh: weighted)
print(f'F1-Score : {f1_score(y_test, pred_base, average="weighted"):.4f}')

# Untuk multiclass, roc_auc_score memerlukan parameter 'multi_class' dan full probabilities
print(f'ROC-AUC  : {roc_auc_score(y_test, proba_base, multi_class="ovr", average="weighted"):.4f}')

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== BASELINE (Multiclass) ===
Accuracy : 0.7055
F1-Score : 0.7005
ROC-AUC  : 0.9127


**Pipeline Konstruksi Data Lengkap**

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 1. Menentukan daftar kolom berdasarkan jenisnya
kolom_num = [
    'age', 'years_experience', 'years_at_company', 'salary_usd',
    'work_hours_per_week', 'meetings_per_day', 'team_size',
    'sleep_hours_per_night', 'exercise_days_per_week', 'vacation_days_taken',
    'therapy_access', 'uses_therapy', 'ai_tools_daily', 'manager_support_score',
    'work_life_balance_score', 'job_satisfaction_score', 'social_support_score',
    'deadline_pressure_score', 'autonomy_score', 'stress_score', 'burnout_score',
    'phq9_score', 'gad7_score', 'seeks_mental_health_support', 'job_change_intention'
]

kolom_kat_ohe = ['gender', 'country', 'job_role', 'work_mode', 'industry']

kolom_kat_ord = ['seniority_level', 'company_size', 'phq9_category', 'gad7_category']

# 2. Membangun Preprocessor dengan ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), kolom_num),
    ('cat_ohe', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), kolom_kat_ohe),
    ('cat_ord', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), kolom_kat_ord),
])

# 3. Membangun Pipeline Lengkap
pipe = Pipeline([
    ('prep', preprocessor),
    ('clf',  LogisticRegression(max_iter=2000, random_state=42))
])

# 4. Training dan Prediksi
pipe.fit(X_train, y_train)
pred_pipe = pipe.predict(X_test)
# Karena multiclass, kita ambil semua kolom probabilitas
proba_pipe = pipe.predict_proba(X_test)

# 5. Evaluasi (Disesuaikan untuk Multiclass)
print('=== PIPELINE KONSTRUKSI LENGKAP ===')
print(f'Accuracy : {accuracy_score(y_test, pred_pipe):.4f}')
print(f'F1-Score : {f1_score(y_test, pred_pipe, average="weighted"):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, proba_pipe, multi_class="ovr", average="weighted"):.4f}')

=== PIPELINE KONSTRUKSI LENGKAP ===
Accuracy : 0.9998
F1-Score : 0.9998
ROC-AUC  : 1.0000
